In [20]:
import os
import glob
import pprint
import pandas
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Blast.Applications import NcbimakeblastdbCommandline
from Bio.Blast.Applications import NcbiblastpCommandline
from Bio.Blast import NCBIXML


Link: [pfam32 reference article, Casjens et al (2012)](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0033280)

In [2]:
assemblies = glob.glob("/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/*.gbff")
print(assemblies)
testing_set_file = 'seqs/Minimal-PF32-testor-set-March2021.faa'
pfam32_name = "parA"

['/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/ESI26.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI112.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI101H.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/ESI26H.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI118H.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI89H.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI33H.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/UNY1032P.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/UCT110.gbff', '/Users/mf019/bioinformatics/longread_GWAS/assembli

In [3]:
# now lets get that file into a single Bio seqrecord dict

def fasta_to_dict(fasta_file):
    fasta_dict = SeqIO.to_dict(SeqIO.parse(fasta_file, "fasta"))
    return(fasta_dict)

def parse_blast_results(blast_results : str) -> dict:
    file_size = os.path.getsize(blast_results)
    if file_size == 0:
        return None
    results = {}
    with open(blast_results, "r") as f:
        blast_records = NCBIXML.parse(f)
        for record in blast_records:
            results[record.query] = []
            for alignment in record.alignments:
                for hsp in alignment.hsps:
                    results[record.query].append((alignment.title, hsp.expect, hsp.score,))
    return results

In [4]:
pf32_dict = fasta_to_dict(testing_set_file)
pf32_db = "pf32_db/pf32"
# now to make blastDB
makeblastdb_cmd = NcbimakeblastdbCommandline(input_file=testing_set_file, dbtype="prot", out=pf32_db)
print(makeblastdb_cmd)
stdout, stderr = makeblastdb_cmd()


makeblastdb -out pf32_db/pf32 -dbtype prot -in seqs/Minimal-PF32-testor-set-March2021.faa


In [5]:
# lets make a dictionary containing our contigs, parse the genbank and pull the top level details for this.
# I forgot that I already did this below but this can hang around in case I need it (I probably will...)
assembly_dict = {}
for file in assemblies:
    print(file.split('/')[-1])
    sample_id = file.split("/")[-1].split(".")[0]
    assembly_dict[sample_id] = {}
    with open(file, "r") as f:
        record = SeqIO.parse(f, "genbank")
        for rec in record:
            contig_id = rec.id
            assembly_dict[sample_id]["contigs"] = {contig_id: {}}
            contig_sequence = rec.seq
            contig_len = len(rec.seq)
            num_feats = len(rec.features)
            assembly_dict[sample_id]["contigs"][contig_id] = {"contig_seq" : contig_sequence, "contig_len": contig_len, "feature_count": num_feats}

ESI26.gbff
URI112.gbff
URI101H.gbff
ESI26H.gbff
URI118H.gbff
URI89H.gbff
URI33H.gbff
UNY1032P.gbff
UCT110.gbff
URI86.gbff
UCT29.gbff
URI103H.gbff
URI107H.gbff
URI44.gbff
URI56H.gbff
URI87.gbff
UNY203P.gbff
URI91.gbff
UCT32.gbff
UWI263.gbff
URI48.gbff
URI33.gbff
URI118.gbff
URI93H.gbff
URI102H.gbff
URI91H.gbff
UNY1090P.gbff
URI102.gbff
URI34.gbff
UNY172.gbff
UWI248.gbff
URI88H.gbff
URI48H.gbff
UCT35.gbff
UNY1128P.gbff
URI34H.gbff
UNY149.gbff
URI120H.gbff
UWI248P.gbff
URI103.gbff
UCT96.gbff
URI42.gbff
URI39.gbff
URI36H.gbff
UNY169.gbff
URI120.gbff
UNY1083P.gbff
UCT30H.gbff
URI41.gbff
URI42H.gbff
UNY193.gbff
UNY193P.gbff
UCT29H.gbff
UCT32H.gbff
URI40H.gbff
URI36.gbff
URI111H.gbff
UCT113H.gbff
URI44H.gbff
URI101.gbff
UNY1085P.gbff
URI117.gbff
UNY208P.gbff
UNY203.gbff
UWI247.gbff
URI86H.gbff
URI40.gbff
URI117H.gbff
URI46H.gbff
URI56.gbff
UCT31.gbff
UCT92H.gbff
UNY1038P.gbff
URI88.gbff
UNY208.gbff
UNY149P.gbff
URI39H.gbff
URI41H.gbff
UCT31H.gbff
UWI263P.gbff
URI47.gbff
UWI283.gbff
URI112H.gb

In [30]:
parA_genes = {}
for file in assemblies:
    print(file)
    sample_id = file.split("/")[-1].split(".")[0]
    parA_genes[sample_id] = []
    with open(file, "r") as f:
        record = SeqIO.parse(f, "genbank")
        for rec in record:
            #print(rec.id)
            for feature in rec.features:
                if feature.type == "CDS":
                    product = feature.qualifiers['product']
                    if "ParA" in product[0]:
                        #print(rec.id, product)
                        feature.qualifiers['product']
                        # get sequence
                        #print(feature.location)
                        # Now lets make a new SeqRecord object for the gene and append it to our list of OspC Genes!
                        nucseq = feature.extract(rec.seq)
                        if len(nucseq) % 3 != 0:
                            print(f"Sequence length not divisible by 3 for {rec.id}---{sample_id}")
                            nucseq += "N"
                            print("adding N!")
                        else:
                            protseq = nucseq.translate(table=11, to_stop=True)
                            gene = SeqIO.SeqRecord(protseq, id=f"{sample_id}-{rec.id}-ParA", name=product[0], description=f"{rec.id}---{sample_id}", dbxrefs=None)
                            parA_genes[sample_id].append(gene)

/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/ESI26.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI112.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI101H.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/ESI26H.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI118H.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI89H.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/URI33H.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/UNY1032P.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/UCT110.gbff
/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotati

In [31]:
pprint.pprint(parA_genes)

{'ESI26': [SeqRecord(seq=Seq('MKIISVINQKGGVGKTTSAINISYSMTLLNKKILLIDIDSQGNSTSGTNTSKHI...IKE'), id='ESI26-contig_1-ParA', name='ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ', description='contig_1---ESI26', dbxrefs=[]),
           SeqRecord(seq=Seq('MDKKETKVITIASIKGGVGKSTTSLIFATLLSIKCKVLLIDIDTQASTTSYFFN...YVH'), id='ESI26-contig_4-ParA', name='ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ', description='contig_4---ESI26', dbxrefs=[]),
           SeqRecord(seq=Seq('MDQKKPIVITLASLKGGVGKSSLSILFSYVLKELGKKVLLIDLDPQNSLTSYFN...NIL'), id='ESI26-contig_5-ParA', name='ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ', description='contig_5---ESI26', dbxrefs=[]),
           SeqRecord(seq=Seq('MDRKKSKIITIASLKGGVGKSTTSIILANLLSKKYKVLLIDTDDQAATTSYYYN...GNL'), id='ESI26-contig_6-ParA', name='ParA-like ATPase involved in chromosome/plasmid partiti

In [8]:
with open('parA_genes_query_v2.faa', 'w') as out:
    for sample in parA_genes:
        for gene in parA_genes[sample]:
            print(gene)
            SeqIO.write(gene, out, "fasta")

ID: ESI26-contig_1-ParA
Name: ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ
Description: contig_1---ESI26
Number of features: 0
Seq('MKIISVINQKGGVGKTTSAINISYSMTLLNKKILLIDIDSQGNSTSGTNTSKHI...IKE')
ID: ESI26-contig_4-ParA
Name: ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ
Description: contig_4---ESI26
Number of features: 0
Seq('MDKKETKVITIASIKGGVGKSTTSLIFATLLSIKCKVLLIDIDTQASTTSYFFN...YVH')
ID: ESI26-contig_5-ParA
Name: ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ
Description: contig_5---ESI26
Number of features: 0
Seq('MDQKKPIVITLASLKGGVGKSSLSILFSYVLKELGKKVLLIDLDPQNSLTSYFN...NIL')
ID: ESI26-contig_6-ParA
Name: ParA-like ATPase involved in chromosome/plasmid partitioning or cellulose biosynthesis protein BcsQ
Description: contig_6---ESI26
Number of features: 0
Seq('MDRKKSKIITIASLKGGVGKSTTSIILANLLSKKYKVLLIDTDDQAATTSYYYN...GNL')


In [9]:
# now lets blast our sequences against the pf32 database we just made!
parA_blast_out = "parA_blast_out_v2.xml"
parA_blast_cmd = NcbiblastpCommandline(query="parA_genes_query_v2.faa", db=pf32_db, out=parA_blast_out, outfmt=5)
print(parA_blast_cmd)
stdout, stderr = parA_blast_cmd()

blastp -out parA_blast_out_v2.xml -outfmt 5 -query parA_genes_query_v2.faa -db pf32_db/pf32


In [10]:
plasmid_content_asm = {}
for file in assemblies:
    print(file)
    sample_id = file.split("/")[-1].split(".")[0]
    parA_genes[sample_id] = []
    plasmid_content_asm[sample_id] = {}
    with open(file, "r") as f:
        record = SeqIO.parse(f, "genbank")
        for rec in record:
            print(rec.id, len(rec.seq))
            plasmid_content_asm[sample_id][rec.id] = {"len" : len(rec.seq), "plasmid" : "", "top_pfam32_hit" : ""}

/Users/mf019/bioinformatics/longread_GWAS/assemblies/paired_assemblies/annotation/genbank/ESI26.gbff
contig_1 469018
contig_2 221524
contig_3 213354
contig_4 53849
contig_5 34908
contig_6 32476
contig_7 27851
contig_8 27297
contig_9 26723
contig_10 26464
contig_11 22619
contig_12 21973
contig_13 11258
contig_14 8371
contig_15 7831
contig_16 6345
contig_17 6020
contig_18 5141
contig_19 5043
contig_20 4606
contig_21 4331
contig_22 4089
contig_23 4042
contig_24 3766
contig_25 3330
contig_26 3234
contig_27 3158
contig_28 2785
contig_29 2390
contig_30 2215
contig_31 2156
contig_32 2019
contig_33 1854
contig_34 1794
contig_35 1543
contig_36 1410
contig_37 1363
contig_38 1326
contig_39 1183
contig_40 1167
contig_41 1011
contig_42 826
contig_43 814
contig_44 757
contig_45 752
contig_46 745
contig_47 734
contig_48 722
contig_49 703
contig_50 702
contig_51 663
contig_52 648
contig_53 644
contig_54 638
contig_55 631
contig_56 589
contig_57 565
contig_58 553
contig_59 530
contig_60 529
contig_61 5

In [11]:
pprint.pprint(plasmid_content_asm)

{'ESI26': {'contig_1': {'len': 469018, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_10': {'len': 26464, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_100': {'len': 297, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_101': {'len': 296, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_102': {'len': 293, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_103': {'len': 288, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_104': {'len': 286, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_105': {'len': 282, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_106': {'len': 282, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_107': {'len': 280, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_108': {'len': 277, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_109': {'len': 277, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_11': {'len': 22619, 'plasmid': '', 'top_pfam32_hit': ''},
         

In [12]:
# now lets parse our blast results
results = parse_blast_results(parA_blast_out)
for query in results:
    #print(query)
    query_id = query.split(" ")
    contig_id = query_id[1].split("---")[0]
    #print(contig_id)
    sample_id = query_id[1].split("---")[1]
    #print(sample_id)
    contig_len = plasmid_content_asm[sample_id][contig_id]["len"]
    #print(query_id, contig_id, sample_id, contig_len)
    hit = results[query][0]
    #print(hit)
    hit_id = hit[0].split(" ")[1].split("_")[-1]
    #print(sample_id, contig_id, hit_id)
    if contig_len > 150000:
        plasmid_content_asm[sample_id][contig_id]["plasmid"] = "chromosome" # this assumption needs work......
    else:
        plasmid_content_asm[sample_id][contig_id]["top_pfam32_hit"] = hit_id

In [13]:
pprint.pprint(plasmid_content_asm)

{'ESI26': {'contig_1': {'len': 469018,
                        'plasmid': 'chromosome',
                        'top_pfam32_hit': ''},
           'contig_10': {'len': 26464, 'plasmid': '', 'top_pfam32_hit': 'cp26'},
           'contig_100': {'len': 297, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_101': {'len': 296, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_102': {'len': 293, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_103': {'len': 288, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_104': {'len': 286, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_105': {'len': 282, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_106': {'len': 282, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_107': {'len': 280, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_108': {'len': 277, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_109': {'len': 277, 'plasmid': '', 'top_pfam32_hit': ''},
           'contig_11': 

In [14]:
plasmids_per_sample = {}
for sample in plasmid_content_asm:
    plasmids_per_sample[sample] = []
    for contig in plasmid_content_asm[sample]:
        #print(sample, contig, plasmid_content_asm[sample][contig]["top_pfam32_hit"], plasmid_content_asm[sample][contig]["len"])
        if plasmid_content_asm[sample][contig]["top_pfam32_hit"] != "":
            plasmids_per_sample[sample].append(plasmid_content_asm[sample][contig]["top_pfam32_hit"])
        else:
            plasmids_per_sample[sample].append(plasmid_content_asm[sample][contig]["plasmid"])

In [15]:
pprint.pprint(plasmids_per_sample)

{'ESI26': ['chromosome',
           '',
           '',
           'lp54',
           'lp36',
           'lp28-4',
           'cp32-10',
           '',
           'lp28-2',
           'cp26',
           '',
           'lp28-3',
           'cp32-1',
           '',
           'cp32-9',
           'lp28-7',
           '',
           '',
           '',
           '',
           '',
           'cp32-6',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',
           '',

In [16]:
all_plasmid_set = set()
for sample in plasmids_per_sample:
    print(sample, set(plasmids_per_sample[sample]), len(set(plasmids_per_sample[sample])))
    all_plasmid_set.update(set(plasmids_per_sample[sample]))
#print(all_plasmid_set, len(all_plasmid_set))

ESI26 {'', 'cp32-1', 'lp28-2', 'lp36', 'chromosome', 'cp32-10', 'cp26', 'lp28-7', 'lp54', 'cp32-6', 'lp28-3', 'lp28-4', 'cp32-9'} 13
URI112 {'', 'lp36', 'lp21', 'chromosome', 'cp32-3', 'cp26', 'lp28-1', 'cp32-8', 'cp32-7', 'lp54', 'cp32-6', 'lp28-3', 'cp32-4', 'lp38', 'lp25', 'lp28-4', 'cp32-9'} 17
URI101H {'', 'lp28-2', 'lp36', 'lp21', 'chromosome', 'cp32-3', 'lp28-1', 'cp26', 'lp56', 'cp32-7', 'lp54', 'cp32-4', 'cp32-6', 'lp38', 'lp25', 'lp28-4', 'lp28-3', 'cp32-9'} 18
ESI26H {'', 'cp32-1', 'lp36', 'chromosome', 'cp32-10', 'cp26', 'lp28-7', 'cp32-6', 'lp28-3', 'lp28-4', 'cp32-9'} 11
URI118H {'', 'lp36', 'chromosome', 'cp32-3', 'cp26', 'lp28-1', 'cp32-4', 'lp54', 'cp32-6', 'lp28-3', 'cp32-7', 'lp28-4'} 12
URI89H {'', 'lp28-2', 'lp21', 'cp32-8', 'lp54', 'lp36', 'cp26', 'cp32-4', 'lp28-4', 'cp32-10', 'cp32-3', 'cp32-7', 'lp28-3', 'cp32-1', 'chromosome', 'lp28-1', 'lp56', 'lp38', 'lp25', 'cp32-9'} 20
URI33H {'', 'lp28-8', 'lp36', 'lp21', 'chromosome', 'cp32-10', 'cp26', 'lp28-7', 'cp32-8

In [17]:
#for sample in plasmid_content_asm:
#    for contig in plasmid_content_asm[sample]:
#        print(sample, contig, plasmid_content_asm[sample][contig]["pfam32_type"], plasmid_content_asm[sample][contig]["len"])

In [18]:
# okay so now we have the pfam32 type for each contig in each sample sort of
# now we need to output this into a table
# sample, contig, pfam32_type, plasmid_length

for sample in plasmid_content_asm:
    with open(f"{sample}_plasmid_content_v2.tsv", "w") as out:
        out.write("contig\tpfam32_type\tplasmid_length\n")
        for contig in plasmid_content_asm[sample]:
            out.write(f"{contig}\t{plasmid_content_asm[sample][contig]['top_pfam32_hit']}\t{plasmid_content_asm[sample][contig]['len']}\n")

In [25]:
# Better yet, let's take that data and make it into a count matrix via pandas where each column is a plasmid with the count indicated for each
# row (sample)

# we already have a set of all plasmids => all_plasmid_set !! We need to convert to a list, pandas does not like sets as columns >:|
all_plasmid_set = list(all_plasmid_set)
# we have a dictionary of plasmids per sample => plasmids_per_sample
# we have everything we need to make this dataframe

df = pandas.DataFrame(columns=all_plasmid_set)
for sample in plasmids_per_sample:
    df.loc[sample] = 0
    for plasmid in plasmids_per_sample[sample]:
        df.at[sample, plasmid] += 1

df.rename(columns={'': "NaN"}, inplace=True)
for index, row in df.iterrows():
    print(index, row)
    df['num_contigs'] = df.cumsum(axis=1).iloc[:, -2]

#df.rename(columns={'index': "sample"}, inplace=True)
df = df.sort_index()
df.to_csv("plasmid_content_matrix_v2.tsv", sep="\t")


ESI26 NaN           150
lp28-2          1
lp21            0
cp32-8          0
lp54            1
lp28-6          0
lp28-8          0
lp36            1
cp26            1
cp32-4          0
cp32-6          1
lp28-4          1
cp32-10         1
cp32-3          0
cp32-7          0
lp28-3          1
cp32-1          1
chromosome      1
lp28-1          0
lp28-7          1
lp56            0
lp38            0
lp25            0
cp32-9          1
Name: ESI26, dtype: int64
URI112 NaN           190
lp28-2          0
lp21            1
cp32-8          1
lp54            1
lp28-6          0
lp28-8          0
lp36            1
cp26            1
cp32-4          1
cp32-6          1
lp28-4          1
cp32-10         0
cp32-3          1
cp32-7          1
lp28-3          1
cp32-1          0
chromosome      1
lp28-1          1
lp28-7          0
lp56            0
lp38            1
lp25            1
cp32-9          1
Name: URI112, dtype: int64
URI101H NaN           15
lp28-2         1
lp21           1
cp32-8     

In [211]:
from Bio import Align
from Bio.Align import substitution_matrices

# code ~~borrowed~~ adapted from github.com/joaorodrigues/seq_align.py
def align_for_distance(sequence1, sequence2):
    """align_for_distance
    This takes two sequences and using biopython.Align.PairwiseAligner(), outputs the identity and gap identity between them.

    Args:
        sequence1 (str): nucleotide sequence for sequence1 (QUERY)
        sequence2 (str): nucleotide sequence for sequence2 (TARGET)
    """
    def _get_identity(sequence1, sequence2):
        #print(seq1, seq2, seqlen)
        seq1, seq2, seqlen = sequence1, sequence2, len(sequence1)
        matches = [seq1[i] == seq2[i] for i in range(seqlen)]
        seq_id = (100 * sum(matches)) / seqlen
        gapless_seqlen = sum([1 for i in range(seqlen) if (seq1[i] != '-' and seq2[i] != '-')])
        gap_id = (100 * sum(matches)) / gapless_seqlen
        return (seq_id, gap_id)

    # set up the aligner
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
    aligner.target_end_gap_score = 0
    aligner.query_end_gap_score = 0

    # align the sequences
    alignments = aligner.align(sequence1, sequence2)
    alignment = alignments[0]
    aln1, aln2= alignment
    seq_id, gap_id = _get_identity(aln1, aln2)
    return seq_id, gap_id

In [212]:
# now we need to make a table of the plasmid content for each sample based on pairwise identity to the pfam32 reference fasta
# we can use Bio.Align.PairwiseAligner for this
# I wrote a function for this that I borrowed from github.com/joaorodrigues/seq_align.py

# we need to make a dictionary of the pfam32 sequences since we do not have that for some reason
pfam32_dict = fasta_to_dict(testing_set_file)

# we need to make a dictionary of the alignments
alignments = {}
for sample in parA_genes:
    #print(sample)
    alignments[sample] = {}
    for gene in parA_genes[sample]:
        alignments[sample][gene.id] = {}
        for pfam32 in pfam32_dict:
            pfam_id = pfam32.split('_')[-1]
            print(sample, gene.id, pfam_id)
            alignments[sample][gene.id][pfam_id] = align_for_distance(str(gene.seq), str(pfam32_dict[pfam32].seq))

ESI26 ESI26-contig_1-ParA DatabaseShortContig
ESI26 ESI26-contig_1-ParA lp32-3#2
ESI26 ESI26-contig_1-ParA lp28-14
ESI26 ESI26-contig_1-ParA lp28-13
ESI26 ESI26-contig_1-ParA lp28-12
ESI26 ESI26-contig_1-ParA lp28-1
ESI26 ESI26-contig_1-ParA lp28-9
ESI26 ESI26-contig_1-ParA lp28-8
ESI26 ESI26-contig_1-ParA lp28-7
ESI26 ESI26-contig_1-ParA lp28-6
ESI26 ESI26-contig_1-ParA lp28-5
ESI26 ESI26-contig_1-ParA lp28-4
ESI26 ESI26-contig_1-ParA lp28-3
ESI26 ESI26-contig_1-ParA lp28-2
ESI26 ESI26-contig_1-ParA lp28-1
ESI26 ESI26-contig_1-ParA lp17
ESI26 ESI26-contig_1-ParA lp21
ESI26 ESI26-contig_1-ParA lp25
ESI26 ESI26-contig_1-ParA lp36
ESI26 ESI26-contig_1-ParA lp38
ESI26 ESI26-contig_1-ParA lp54
ESI26 ESI26-contig_1-ParA lp56
ESI26 ESI26-contig_1-ParA cp9
ESI26 ESI26-contig_1-ParA cp26
ESI26 ESI26-contig_1-ParA cp32-1
ESI26 ESI26-contig_1-ParA cp32-3
ESI26 ESI26-contig_1-ParA cp32-4
ESI26 ESI26-contig_1-ParA cp32-5
ESI26 ESI26-contig_1-ParA cp32-6
ESI26 ESI26-contig_1-ParA cp32-7
ESI26 ESI26

In [213]:
pf32_hits_per_sample = {}
for sample in alignments:
    pf32_hits_per_sample[sample] = {}
    for gene in alignments[sample]:
        contig_id = gene.split("-")[1]
        pf32_hits_per_sample[sample][contig_id] = {}
        for pfam32 in alignments[sample][gene]:
            pfam32_id = pfam32.split("_")[-1]
            #print(sample, contig_id, pfam32, alignments[sample][gene][pfam32],)# alignments[sample][gene][pfam32].distance)
            pf32_hits_per_sample[sample][contig_id][pfam32_id] = alignments[sample][gene][pfam32][1]

In [214]:
pf32_hits_per_sample['ESI26']

{'contig_1': {'DatabaseShortContig': 34.13461538461539,
  'lp32-3#2': 34.61538461538461,
  'lp28-14': 35.267857142857146,
  'lp28-13': 37.66816143497758,
  'lp28-12': 39.189189189189186,
  'lp28-1': 32.900432900432904,
  'lp28-9': 34.49781659388646,
  'lp28-8': 37.5,
  'lp28-7': 36.986301369863014,
  'lp28-6': 30.76923076923077,
  'lp28-5': 38.11659192825112,
  'lp28-4': 41.39784946236559,
  'lp28-3': 28.755364806866954,
  'lp28-2': 40.52863436123348,
  'lp17': 28.389830508474578,
  'lp21': 35.45454545454545,
  'lp25': 38.11659192825112,
  'lp36': 35.648148148148145,
  'lp38': 39.23444976076555,
  'lp54': 36.69724770642202,
  'lp56': 38.05309734513274,
  'cp9': 37.27272727272727,
  'cp26': 34.1991341991342,
  'cp32-1': 36.03603603603604,
  'cp32-3': 38.3177570093458,
  'cp32-4': 30.93220338983051,
  'cp32-5': 37.78801843317972,
  'cp32-6': 35.74660633484163,
  'cp32-7': 37.96296296296296,
  'cp32-8': 40.84507042253521,
  'cp32-9': 36.32286995515695,
  'cp32-10': 38.78504672897196,
  'c

In [192]:
pairwise_dataframes = {k: pandas.DataFrame(columns=all_plasmid_set) for k in plasmid_content_asm.keys()}
for sample in plasmid_content_asm:
    print(sample)
    for contig in plasmid_content_asm[sample]:
        if contig not in pf32_hits_per_sample[sample]:
            print(contig)
            pairwise_dataframes[sample].loc[contig,:] = 0
        else:
            for pf32 in pf32_hits_per_sample[sample][contig]:
                pairwise_dataframes[sample].loc[contig, pf32] = pf32_hits_per_sample[sample][contig][pf32]

ESI26
contig_2
contig_3
contig_8
contig_11
contig_14
contig_17
contig_18
contig_19
contig_20
contig_21
contig_23
contig_24
contig_25
contig_26
contig_27
contig_28
contig_29
contig_30
contig_31
contig_32
contig_33
contig_34
contig_35
contig_36
contig_37
contig_38
contig_39
contig_40
contig_41
contig_42
contig_43
contig_44
contig_45
contig_46
contig_47
contig_48
contig_49
contig_50
contig_51
contig_52
contig_53
contig_54
contig_55
contig_56
contig_57
contig_58
contig_59
contig_60
contig_61
contig_62
contig_63
contig_64
contig_65
contig_66
contig_67
contig_68
contig_69
contig_70
contig_71
contig_72
contig_73
contig_74
contig_75
contig_76
contig_77
contig_78
contig_79
contig_80
contig_81
contig_82
contig_83
contig_84
contig_85
contig_86
contig_87
contig_88
contig_89
contig_90
contig_91
contig_92
contig_93
contig_94
contig_95
contig_96
contig_97
contig_98
contig_99
contig_100
contig_101
contig_102
contig_103
contig_104
contig_105
contig_106
contig_107
contig_108
contig_109
contig_110
contig

In [193]:
for sample in pf32_hits_per_sample:
    for contig in pf32_hits_per_sample[sample]:
        print(sample, contig, pf32_hits_per_sample[sample][contig])

ESI26 contig_1 {'DatabaseShortContig': 10.0, 'lp32-3#2': 3.6, 'lp28-14': 6.8, 'lp28-13': 10.8, 'lp28-12': 8.0, 'lp28-1': 9.6, 'lp28-9': 9.6, 'lp28-8': 8.8, 'lp28-7': 7.6, 'lp28-6': 8.8, 'lp28-5': 8.16326530612245, 'lp28-4': 4.8, 'lp28-3': 10.0, 'lp28-2': 6.0, 'lp17': 15.853658536585366, 'lp21': 10.0, 'lp25': 6.8, 'lp36': 9.236947791164658, 'lp38': 6.122448979591836, 'lp54': 7.2, 'lp56': 10.4, 'cp9': 9.34959349593496, 'cp26': 12.0, 'cp32-1': 8.94308943089431, 'cp32-3': 6.097560975609756, 'cp32-4': 6.4, 'cp32-5': 7.2, 'cp32-6': 6.42570281124498, 'cp32-7': 7.228915662650603, 'cp32-8': 7.317073170731708, 'cp32-9': 8.8, 'cp32-10': 7.2, 'cp32-11': 6.4, 'cp32-12': 8.0, 'cp32-13': 9.6}
ESI26 contig_4 {'DatabaseShortContig': 12.8, 'lp32-3#2': 6.8, 'lp28-14': 12.4, 'lp28-13': 14.0, 'lp28-12': 15.2, 'lp28-1': 14.0, 'lp28-9': 12.8, 'lp28-8': 31.6, 'lp28-7': 11.6, 'lp28-6': 15.6, 'lp28-5': 21.224489795918366, 'lp28-4': 26.8, 'lp28-3': 13.2, 'lp28-2': 7.6, 'lp17': 7.723577235772358, 'lp21': 11.6, 'l

In [217]:
all_asm_pwdf = pandas.DataFrame(columns=all_plasmid_set)
for sample in pairwise_dataframes:
    newrow = pairwise_dataframes[sample].max(axis=0)
    all_asm_pwdf.loc[sample] = newrow
all_asm_pwdf.drop(columns=['chromosome'])
all_asm_pwdf.sort_index(inplace=True)
all_asm_pwdf

,lp28-2,lp21,cp32-8,lp54,lp28-6,lp28-8,lp36,cp26,cp32-4,cp32-6,...,cp32-7,lp28-3,cp32-1,chromosome,lp28-1,lp28-7,lp56,lp38,lp25,cp32-9
ESI26,92.156863,13.438735,74.796748,100.0,72.50996,39.2,100.0,99.604743,58.964143,99.598394,...,64.658635,97.6,100.0,0,56.521739,96.047431,68.525896,46.530612,56.097561,100.0
ESI26H,9.236948,13.438735,74.796748,60.4,72.50996,39.2,100.0,99.604743,58.964143,99.598394,...,64.658635,97.6,100.0,0,56.521739,96.047431,68.525896,46.530612,56.097561,100.0
UCT109,100.0,13.438735,73.98374,100.0,71.713147,38.8,100.0,99.604743,100.0,100.0,...,100.0,100.0,64.227642,0,100.0,57.768924,95.256917,100.0,100.0,100.0
UCT109H,10.714286,13.438735,73.98374,100.0,71.713147,38.8,100.0,99.604743,100.0,100.0,...,100.0,100.0,64.227642,0,60.557769,57.768924,67.729084,100.0,100.0,100.0
UCT110,10.714286,99.618321,100.0,100.0,71.713147,38.8,100.0,98.418972,98.804781,99.598394,...,100.0,100.0,64.227642,0,99.606299,57.370518,67.729084,91.836735,98.412698,99.601594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UWI248P,10.714286,13.043478,63.00813,100.0,71.713147,38.8,100.0,98.418972,98.804781,99.598394,...,100.0,100.0,99.593496,0,99.606299,57.370518,67.729084,91.836735,98.412698,48.605578
UWI263,10.714286,13.14741,60.569106,100.0,71.713147,38.8,100.0,48.605578,60.956175,99.598394,...,64.634146,100.0,64.227642,0,99.606299,55.731225,67.729084,91.836735,98.412698,99.601594
UWI263P,10.714286,13.14741,100.0,100.0,71.713147,38.8,100.0,98.418972,60.956175,99.598394,...,64.634146,100.0,64.227642,0,99.606299,55.731225,67.729084,91.836735,98.412698,99.601594
UWI283,98.431373,13.14741,61.382114,100.0,72.111554,38.8,93.975904,98.418972,99.203187,32.113821,...,64.634146,99.6,100.0,0,60.956175,56.972112,68.12749,47.346939,99.603175,99.601594


In [196]:
all_asm_pwdf.to_csv("pf32_identity_matrix_v1.tsv", sep="\t")